# Mk33 memory study: more models

Runs the same RAG vs KV-cache comparison as `mk33_rag_vs_kv.ipynb` on four more open models, so the results can be compared across model families and sizes. Same corpus, windows, budgets and settings as the Qwen2.5-3B and Llama-3.2-3B runs.

**Before starting:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all.

Each model takes roughly 15 to 30 minutes on a T4. Results download after **each** model, so a disconnect does not lose finished work; rerunning skips finished measurements as long as the session's files are still there. Models are independent: skip any cell you do not want.

## Setup

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
print(torch.cuda.get_device_name())
%cd /content
!test -d Mk33 || git clone -q https://github.com/dulguuntsatsralt-cell/Mk33
%cd /content/Mk33
!git pull -q
!pip install -q -e '.[mk33-kv,dev]'
!pip install -q -U 'transformers>=4.51'
!git log --oneline -1

Qwen3 needs a recent `transformers`, which the last install line provides.

## Qwen2.5-1.5B-Instruct

Smaller Qwen2.5: does model size change the ranking?

In [ ]:
!python scripts/mk33/memory_quality.py --model Qwen/Qwen2.5-1.5B-Instruct --dtype float16 \
    --out results/memq_qwen2.5-1.5b 2>&1 | grep -v -i warning | grep -v "Loading weights"
!cd /content/Mk33 && zip -qr /content/memq_qwen2.5-1.5b.zip results/memq_qwen2.5-1.5b
from google.colab import files
files.download("/content/memq_qwen2.5-1.5b.zip")

## Llama-3.2-1B-Instruct

Smaller Llama 3.2 (ungated copy).

In [ ]:
!python scripts/mk33/memory_quality.py --model unsloth/Llama-3.2-1B-Instruct --dtype float16 \
    --out results/memq_llama3.2-1b 2>&1 | grep -v -i warning | grep -v "Loading weights"
!cd /content/Mk33 && zip -qr /content/memq_llama3.2-1b.zip results/memq_llama3.2-1b
from google.colab import files
files.download("/content/memq_llama3.2-1b.zip")

## Qwen3-4B

Qwen3 dropped Qwen2.5's biased query/key projections: does key-norm eviction work here?

In [ ]:
!python scripts/mk33/memory_quality.py --model Qwen/Qwen3-4B --dtype float16 \
    --out results/memq_qwen3-4b 2>&1 | grep -v -i warning | grep -v "Loading weights"
!cd /content/Mk33 && zip -qr /content/memq_qwen3-4b.zip results/memq_qwen3-4b
from google.colab import files
files.download("/content/memq_qwen3-4b.zip")

## granite-3.1-2b-instruct

IBM Granite: a different model family.

In [ ]:
!python scripts/mk33/memory_quality.py --model ibm-granite/granite-3.1-2b-instruct --dtype float16 \
    --out results/memq_granite3.1-2b 2>&1 | grep -v -i warning | grep -v "Loading weights"
!cd /content/Mk33 && zip -qr /content/memq_granite3.1-2b.zip results/memq_granite3.1-2b
from google.colab import files
files.download("/content/memq_granite3.1-2b.zip")

## Check for numerical problems

float16 can overflow on some models. Any `nan` or `inf` below means that model's results are not usable as they stand.

In [ ]:
import json, glob, math
for f in sorted(glob.glob("results/memq_*/rows.jsonl")):
    rows = [json.loads(l) for l in open(f)]
    bad = sum(1 for r in rows if not math.isfinite(r["nll"]))
    print(f"{f}: {len(rows)} measurements, {bad} not finite")